<a href="https://colab.research.google.com/github/Mobeen-munir/Final-Project-Generative-Ai/blob/main/procodeeee_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# No hard version pins, no forced torch/CUDA reinstall, no xformers.
# Colab already ships a working torch+CUDA+numpy stack -- forcing old pinned
# versions (e.g. xformers==0.0.27 + an explicit torch index) is what was dragging
# in an old numpy/websockets combo and breaking other preinstalled packages.
# Letting pip resolve against what is already installed avoids that entirely.
!pip install -q -U diffusers transformers accelerate controlnet-aux sentence-transformers gradio safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 which is incompatible.


In [2]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import re

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from controlnet_aux import OpenposeDetector
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor
import gradio as gr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via t

Using device: cuda


In [3]:
def parse_script(script_text):
    """Split raw script text into a list of panel dicts: [{"panel_id": int, "action_text": str}]"""
    lines = [l.strip() for l in script_text.strip().split("\n") if l.strip()]
    panels = [{"panel_id": i + 1, "action_text": line} for i, line in enumerate(lines)]
    return panels

# quick self-test
_test = parse_script("A boy sits quietly reading a book.\nHe suddenly leaps up and sprints out the door.")
print(_test)


[{'panel_id': 1, 'action_text': 'A boy sits quietly reading a book.'}, {'panel_id': 2, 'action_text': 'He suddenly leaps up and sprints out the door.'}]


In [4]:
import re
import numpy as np
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer

ACTION_WORDS = {
    "run", "running", "sprint", "sprinted", "jump", "jumping", "leap", "leapt", "fight", "fighting",
    "punch", "kick", "chase", "chasing", "explode", "explosion", "fall", "falling", "scream", "shout",
    "attack", "battle", "dash", "race", "swing", "throw", "crash", "burst", "flee", "escape",
}
CALM_WORDS = {
    "sit", "sitting", "stand", "standing", "talk", "talking", "think", "thinking", "smile", "smiling",
    "whisper", "rest", "sleep", "read", "reading", "wait", "waiting", "watch", "watching", "look",
    "gaze", "stare", "calm", "quiet", "still",
}

def heuristic_intensity(text):
    """Returns a 0-1 action-intensity score based on keyword overlap."""
    words = set(re.findall(r"[a-zA-Z']+", text.lower()))
    action_hits = len(words & ACTION_WORDS)
    calm_hits = len(words & CALM_WORDS)
    if action_hits == 0 and calm_hits == 0:
        return 0.4  # neutral default when no keyword matches
    score = action_hits / (action_hits + calm_hits + 1e-6)
    return float(np.clip(score, 0.0, 1.0))

def heuristic_weights(text):
    """Maps intensity -> (identity_weight, pose_w).
    High intensity -> lower identity rigidity, higher pose freedom.
    Low intensity (calm) -> higher identity fidelity, lower pose freedom needed."""
    intensity = heuristic_intensity(text)
    identity_w = float(np.clip(0.95 - 0.45 * intensity, 0.3, 0.95))
    pose_w = float(np.clip(0.25 + 0.65 * intensity, 0.2, 0.9))
    return identity_w, pose_w


class GatingMLP(nn.Module):
    """Small MLP: sentence embedding (384-d, from all-MiniLM-L6-v2) -> (identity_w, pose_w) in (0,1)."""
    def __init__(self, in_dim=384, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 2), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


# Optional: Set seed for reproducible MLP initial weights
torch.manual_seed(42)

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
gating_model = GatingMLP().to(DEVICE)

# --- Weak-supervision bootstrap training ---
TRAIN_SENTENCES = [
    "She sits by the window, quietly reading a letter.",
    "He whispers a secret to his friend in the hallway.",
    "They stand still, watching the sunset in silence.",
    "The old man rests on a bench, thinking about the past.",
    "She gazes at the stars, lost in thought.",
    "He calmly pours a cup of tea.",
    "The cat sleeps peacefully on the windowsill.",
    "She smiles and waves at the crowd.",
    "He walks slowly down the empty street.",
    "They talk quietly over dinner.",
    "She sprints down the alley, heart pounding.",
    "He leaps over the fence, narrowly escaping.",
    "The two heroes clash swords in a fierce battle.",
    "An explosion rips through the building as she dives for cover.",
    "He throws a punch and the crowd gasps.",
    "She bolts across the rooftop, chased by shadows.",
    "The car crashes through the barrier in a burst of sparks.",
    "He swings wildly, narrowly missing his opponent.",
    "They race through the burning building, screaming for help.",
    "She kicks the door open and charges inside.",
]

def bootstrap_train(model, sentences, epochs=200, lr=1e-3):
    # Fix: Explicit conversion alignment to avoid mixed CPU/GPU device constraints
    embeddings = embedder.encode(sentences, convert_to_tensor=True)
    embeddings = embeddings.to(DEVICE, dtype=torch.float32).clone() # Clone to ensure it's not an inference tensor

    pseudo_labels = torch.tensor(
        [heuristic_weights(s) for s in sentences], dtype=torch.float32
    ).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    model.train()

    for epoch in range(epochs):
        opt.zero_grad()
        preds = model(embeddings)
        loss = loss_fn(preds, pseudo_labels)
        loss.backward()
        opt.step()

    model.eval()
    print(f"Gating MLP bootstrap training done. Final loss: {loss.item():.4f}")
    return model

gating_model = bootstrap_train(gating_model, TRAIN_SENTENCES)

def adaptive_weights(text):
    """Main entry point used by the pipeline: semantic, trained version."""
    with torch.no_grad():
        emb = embedder.encode([text], convert_to_tensor=True)
        emb = emb.to(DEVICE, dtype=torch.float32)
        identity_w, pose_w = gating_model(emb)[0].tolist()
    return float(identity_w), float(pose_w)

# quick check
print("heuristic:", heuristic_weights("She bolted across the rooftop, chased by shadows."))
print("adaptive :", adaptive_weights("She bolted across the rooftop, chased by shadows."))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Gating MLP bootstrap training done. Final loss: 0.0001
heuristic: (0.7699999999999999, 0.51)
adaptive : (0.7606525421142578, 0.5171498656272888)


In [5]:
import torch
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from controlnet_aux import OpenposeDetector

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 # Define DTYPE for torch_dtype

# 1. Pipeline and Model Initializations
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose", torch_dtype=DTYPE
).to(DEVICE)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,
).to(DEVICE)

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Load the IP-Adapter weights safely
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")

# 2. Detector Initialization
openpose_detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet").to(DEVICE)

_BLANK_POSE = Image.new("RGB", (512, 512), (128, 128, 128))  # neutral control image

def generate_panel(character_ref_image, pose_ref_image, action_text, identity_w, pose_w,
                    steps=30, guidance_scale=7.5, seed=None):

    # Pre-process your incoming images into canonical sizes
    if character_ref_image is not None:
        character_ref_image = character_ref_image.convert("RGB").resize((512, 512))

    if pose_ref_image is not None:
        pose_input = pose_ref_image.convert("RGB").resize((512, 512))
        try:
            # Universal fallback for OpenPose versions
            control_image = openpose_detector(pose_input)
        except Exception:
            control_image = openpose_detector(pose_input, detect_resolution=512, image_resolution=512)
        conditioning_scale = float(pose_w)
    else:
        control_image = _BLANK_POSE
        conditioning_scale = 0.0

    # Ensure cross-attention weights match strict floating points
    pipe.set_ip_adapter_scale(float(identity_w))

    if seed is not None:
        generator = torch.Generator(device=DEVICE).manual_seed(int(seed))
    else:
        generator = torch.Generator(device=DEVICE).manual_seed(42)

    # Render sequence execution block
    output = pipe(
        prompt=f"comic book panel, {str(action_text)}, dynamic illustration, clean line art",
        negative_prompt="blurry, deformed, extra limbs, watermark, text, low quality, photorealistic",
        image=control_image,
        ip_adapter_image=character_ref_image,
        controlnet_conditioning_scale=conditioning_scale,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance_scale),
        generator=generator,
    )

    return output.images[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

...

models/image_encoder/model.safetensors:   0%|          | 0.00/2.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

annotator/ckpts/body_pose_model.pth:   0%|          | 0.00/209M [00:00<?, ?B/s]

annotator/ckpts/hand_pose_model.pth:   0%|          | 0.00/147M [00:00<?, ?B/s]

facenet.pth:   0%|          | 0.00/154M [00:00<?, ?B/s]

In [6]:
def assemble_comic_strip(panel_images, cols=2, gutter=16, bg_color=(255, 255, 255)):
    if not panel_images:
        return None
    w, h = panel_images[0].size
    rows = int(np.ceil(len(panel_images) / cols))
    strip_w = cols * w + (cols + 1) * gutter
    strip_h = rows * h + (rows + 1) * gutter
    strip = Image.new("RGB", (strip_w, strip_h), bg_color)
    draw = ImageDraw.Draw(strip)

    for i, panel in enumerate(panel_images):
        row, col = divmod(i, cols)
        x = gutter + col * (w + gutter)
        y = gutter + row * (h + gutter)
        strip.paste(panel, (x, y))
        draw.rectangle([x, y, x + w, y + h], outline=(0, 0, 0), width=3)
        draw.text((x + 8, y + 8), str(i + 1), fill=(0, 0, 0))
    return strip


In [7]:
from transformers import CLIPModel, CLIPProcessor
import torch # Import torch to use torch.cuda.is_available()
import numpy as np # Import numpy for np.mean

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def compute_identity_consistency(ref_image, panel_images):
    """Mean CLIP cosine similarity between the reference character image and each generated panel.
    Higher = more consistent identity across panels."""
    imgs = [ref_image] + list(panel_images)
    inputs = clip_processor(images=imgs, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        feats = clip_model.get_image_features(**inputs)
    # transformers >=5.0 wraps this in a BaseModelOutputWithPooling instead of
    # returning a raw tensor -- unwrap it so .norm() works on both old and new versions
    if not torch.is_tensor(feats):
        feats = getattr(feats, "image_embeds", None) or getattr(feats, "pooler_output", None)
        if feats is None:
            raise TypeError("Unexpected output type from get_image_features(); could not find a tensor to unwrap.")
    feats = feats / feats.norm(dim=-1, keepdim=True)
    ref_feat = feats[0]
    sims = [torch.dot(ref_feat, f).item() for f in feats[1:]]
    return float(np.mean(sims))

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
def run_pipeline(character_ref, script_text, pose_ref_files, mode, manual_identity, manual_pose,
                  progress=gr.Progress()):
    if character_ref is None:
        return None, None, "Error: Please upload a Character Reference Image.", "Missing Input"
    if not script_text.strip():
        return None, None, "Error: Please enter script text.", "Missing Input"

    try:
        panels = parse_script(script_text)
        pose_refs = []
        if pose_ref_files:
            for f in pose_ref_files:
                pose_refs.append(Image.open(f.name).convert("RGB"))

        panel_images = []
        weight_lines = []

        for i, panel in enumerate(panels):
            progress((i) / len(panels), desc=f"Generating panel {i + 1}/{len(panels)}")
            if mode == "Adaptive (ours)":
                identity_w, pose_w = adaptive_weights(panel["action_text"])
            else:
                identity_w, pose_w = float(manual_identity), float(manual_pose)

            pose_ref = pose_refs[i] if i < len(pose_refs) else None

            # Call generation function
            img = generate_panel(character_ref, pose_ref, panel["action_text"], identity_w, pose_w)
            panel_images.append(img)
            weight_lines.append(
                f"Panel {i + 1}: identity={identity_w:.2f}, pose={pose_w:.2f} -- \"{panel['action_text']}\""
            )

        strip = assemble_comic_strip(panel_images)
        score = compute_identity_consistency(character_ref, panel_images)
        metric_text = f"Mean identity-consistency (CLIP-I): {score:.3f}"

        # Safe parsing formatting for Gradio Gallery components
        gallery_output = [(img, f"Panel {i+1}") for i, img in enumerate(panel_images)]

        return gallery_output, strip, "\n".join(weight_lines), metric_text

    except Exception as e:
        import traceback
        error_details = traceback.format_exc()
        print(error_details) # Prints the exact line numbers to your Colab console
        return None, None, f"Runtime pipeline error:\n{str(e)}", "Generation Failed"

with gr.Blocks(title="ComicConsist") as demo:
    gr.Markdown("# ComicConsist")
    gr.Markdown("Adaptive identity/pose blending for character-consistent comic panel generation.")

    with gr.Row():
        with gr.Column():
            character_ref = gr.Image(label="Character reference image", type="pil")
            script_text = gr.Textbox(
                label="Script (one panel per line)",
                lines=5,
                value="A boy stands quietly in his room, thinking.\nHe suddenly sprints down the stairs."
            )
            pose_ref_files = gr.File(
                label="Optional: pose reference images",
                file_count="multiple",
            )
            mode = gr.Radio(
                ["Adaptive (ours)", "Fixed baseline"], value="Adaptive (ours)", label="Blending mode"
            )
            with gr.Row():
                manual_identity = gr.Slider(0, 1, value=0.8, label="Fixed identity weight")
                manual_pose = gr.Slider(0, 1, value=0.5, label="Fixed pose weight")
            run_btn = gr.Button("Generate comic", variant="primary")
        with gr.Column():
            gallery = gr.Gallery(label="Generated panels", columns=2)
            strip_out = gr.Image(label="Assembled comic strip")
            weights_out = gr.Textbox(label="Computed blend weights per panel", lines=4)
            metric_out = gr.Textbox(label="Evaluation metric")

    run_btn.click(
        run_pipeline,
        inputs=[character_ref, script_text, pose_ref_files, mode, manual_identity, manual_pose],
        outputs=[gallery, strip_out, weights_out, metric_out],
    )

demo.launch(inline=True, share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8fa2561d785a283503.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]